In [1]:
import io
import os
import urllib.request
import gradio as gr
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter, ImageOps
from scipy.ndimage import center_of_mass, shift

matplotlib.use("Agg")

WEIGHTS_FILE = "mnist_weights.npz"
MNIST_RAW_FILE = "mnist_dataset.npz"

d:\Project\Git_Hub\Ideas\Handwritten-Digit-Recognition-using-ANN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. LOAD / TRAIN PURE NUMPY ANN ON FULL MNIST (70,000 IMAGES)
# ==========================================
def load_mnist_direct():
    """Downloads official MNIST .npz directly without OpenML parser issues."""
    if not os.path.exists(MNIST_RAW_FILE):
        url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
        print("Downloading MNIST dataset (11.5 MB)...")
        urllib.request.urlretrieve(url, MNIST_RAW_FILE)

    with np.load(MNIST_RAW_FILE) as data:
        X_train = data["x_train"].reshape(-1, 784).astype(np.float32) / 255.0
        y_train = data["y_train"].astype(np.int64)
        X_test = data["x_test"].reshape(-1, 784).astype(np.float32) / 255.0
        y_test = data["y_test"].astype(np.int64)
    return X_train, y_train, X_test, y_test


def get_trained_weights():
    if os.path.exists(WEIGHTS_FILE):
        print("Loading pre-trained 784->128->64->10 MNIST weights...")
        data = np.load(WEIGHTS_FILE)
        return (
            data["W1"],
            data["b1"],
            data["W2"],
            data["b2"],
            data["W3"],
            data["b3"],
            float(data["test_acc"]),
            float(data["final_loss"]),
            data["sample_x"],
        )

    X_train, y_train, X_test, y_test = load_mnist_direct()
    Y_train_oh = np.eye(10, dtype=np.float32)[y_train]

    # Data Augmentation: Add shifted versions so off-center mouse strokes never confuse digits
    print("Augmenting training data (300,000 samples)...")
    imgs_2d = X_train.reshape(-1, 28, 28)
    shift_left = np.roll(imgs_2d, -2, axis=2).reshape(-1, 784)
    shift_right = np.roll(imgs_2d, 2, axis=2).reshape(-1, 784)
    shift_up = np.roll(imgs_2d, -2, axis=1).reshape(-1, 784)
    shift_down = np.roll(imgs_2d, 2, axis=1).reshape(-1, 784)

    X_aug = np.vstack([X_train, shift_left, shift_right, shift_up, shift_down])
    Y_aug = np.vstack([Y_train_oh] * 5)

    # He Initialization: 784 -> 128 -> 64 -> 10
    np.random.seed(42)
    W1 = np.random.randn(784, 128).astype(np.float32) * np.sqrt(2.0 / 784)
    b1 = np.zeros(128, dtype=np.float32)
    W2 = np.random.randn(128, 64).astype(np.float32) * np.sqrt(2.0 / 128)
    b2 = np.zeros(64, dtype=np.float32)
    W3 = np.random.randn(64, 10).astype(np.float32) * np.sqrt(2.0 / 64)
    b3 = np.zeros(10, dtype=np.float32)

    lr = 0.08
    momentum = 0.9
    batch_size = 256
    epochs = 12

    vW1, vb1 = np.zeros_like(W1), np.zeros_like(b1)
    vW2, vb2 = np.zeros_like(W2), np.zeros_like(b2)
    vW3, vb3 = np.zeros_like(W3), np.zeros_like(b3)

    n_samples = len(X_aug)
    final_loss = 0.0

    print("Training 3-Layer ANN (784 -> 128 -> 64 -> 10) in pure NumPy...")
    for epoch in range(epochs):
        perm = np.random.permutation(n_samples)
        X_shuf = X_aug[perm]
        Y_shuf = Y_aug[perm]
        epoch_loss = 0.0
        batches = 0

        for i in range(0, n_samples, batch_size):
            xb = X_shuf[i : i + batch_size]
            yb = Y_shuf[i : i + batch_size]
            m = len(xb)

            # Forward Pass
            z1 = xb @ W1 + b1
            h1 = np.maximum(0, z1)
            z2 = h1 @ W2 + b2
            h2 = np.maximum(0, z2)
            z3 = h2 @ W3 + b3
            exp_z3 = np.exp(z3 - np.max(z3, axis=1, keepdims=True))
            probs = exp_z3 / np.sum(exp_z3, axis=1, keepdims=True)

            loss = -np.mean(np.sum(yb * np.log(probs + 1e-8), axis=1))
            epoch_loss += loss
            batches += 1

            # Backpropagation
            dz3 = (probs - yb) / m
            dW3 = h2.T @ dz3
            db3 = np.sum(dz3, axis=0)

            dz2 = (dz3 @ W3.T) * (z2 > 0)
            dW2 = h1.T @ dz2
            db2 = np.sum(dz2, axis=0)

            dz1 = (dz2 @ W2.T) * (z1 > 0)
            dW1 = xb.T @ dz1
            db1 = np.sum(dz1, axis=0)

            # Momentum Update
            vW3 = momentum * vW3 - lr * dW3
            vb3 = momentum * vb3 - lr * db3
            vW2 = momentum * vW2 - lr * dW2
            vb2 = momentum * vb2 - lr * db2
            vW1 = momentum * vW1 - lr * dW1
            vb1 = momentum * vb1 - lr * db1

            W3 += vW3
            b3 += vb3
            W2 += vW2
            b2 += vb2
            W1 += vW1
            b1 += vb1

        final_loss = epoch_loss / batches
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {final_loss:.4f}")

    # Evaluate on 10,000 unseen test digits
    h1_t = np.maximum(0, X_test @ W1 + b1)
    h2_t = np.maximum(0, h1_t @ W2 + b2)
    z3_t = h2_t @ W3 + b3
    test_acc = float(np.mean(np.argmax(z3_t, axis=1) == y_test) * 100)
    print(f"Training Complete! Test Accuracy: {test_acc:.2f}%")

    sample_9 = X_test[y_test == 9][0].reshape(28, 28)
    np.savez(
        WEIGHTS_FILE,
        W1=W1,
        b1=b1,
        W2=W2,
        b2=b2,
        W3=W3,
        b3=b3,
        test_acc=test_acc,
        final_loss=final_loss,
        sample_x=sample_9,
    )
    return W1, b1, W2, b2, W3, b3, test_acc, final_loss, sample_9


W1, b1, W2, b2, W3, b3, TEST_ACC, FINAL_LOSS, SAMPLE_9 = get_trained_weights()

Augmenting training data (300,000 samples)...
Training 3-Layer ANN (784 -> 128 -> 64 -> 10) in pure NumPy...
Epoch 1/12 - Loss: 0.2274
Epoch 2/12 - Loss: 0.0953
Epoch 3/12 - Loss: 0.0718
Epoch 4/12 - Loss: 0.0596
Epoch 5/12 - Loss: 0.0513
Epoch 6/12 - Loss: 0.0445
Epoch 7/12 - Loss: 0.0402
Epoch 8/12 - Loss: 0.0364
Epoch 9/12 - Loss: 0.0331
Epoch 10/12 - Loss: 0.0304
Epoch 11/12 - Loss: 0.0279
Epoch 12/12 - Loss: 0.0263
Training Complete! Test Accuracy: 98.28%


In [3]:
# ==========================================
# 2. THICKENED STROKE & CENTER-OF-MASS PREPROCESSING
# ==========================================
def preprocess_drawing(sketch_data):
    if sketch_data is None or sketch_data.get("composite") is None:
        return None

    img = Image.fromarray(sketch_data["composite"]).convert("L")
    img = ImageOps.invert(img)

    # Thicken strokes so lines remain bold after shrinking to 28x28
    img = img.filter(ImageFilter.MaxFilter(size=5))
    img = img.filter(ImageFilter.GaussianBlur(radius=1.5))

    bbox = img.getbbox()
    if bbox is None:
        return None

    cropped = img.crop(bbox)
    w, h = cropped.size
    if w > h:
        new_w, new_h = 20, max(1, int(round((h / w) * 20)))
    else:
        new_h, new_w = 20, max(1, int(round((w / h) * 20)))

    resized = cropped.resize((new_w, new_h), Image.Resampling.LANCZOS)

    canvas = Image.new("L", (28, 28), 0)
    canvas.paste(resized, ((28 - new_w) // 2, (28 - new_h) // 2))
    arr = np.array(canvas, dtype=np.float32)

    if arr.max() > 0:
        arr = arr / arr.max()

    # Boost pixel intensity for cleaner signal
    arr = np.clip(arr * 1.25, 0.0, 1.0)

    # Center-of-Mass alignment to (14, 14)
    cy, cx = center_of_mass(arr)
    if not np.isnan(cy) and not np.isnan(cx):
        arr = shift(arr, shift=[14.0 - cy, 14.0 - cx], mode="constant", cval=0.0)

    return np.clip(arr, 0.0, 1.0)


def forward_pass(input_28x28):
    x_vec = input_28x28.flatten().reshape(1, 784)
    h1 = np.maximum(0, x_vec @ W1 + b1)[0]
    h2 = np.maximum(0, h1 @ W2 + b2)
    z3 = h2 @ W3 + b3
    exp_z3 = np.exp(z3 - np.max(z3))
    probs = exp_z3 / np.sum(exp_z3)
    return x_vec[0], h1, h2, probs

In [4]:
# ==========================================
# 3. LIVE NEURAL FIRING VISUALIZER
# ==========================================
def render_network_visual(input_28x28, x_vec, h1, h2, probs):
    pred_digit = int(np.argmax(probs))
    confidence = probs[pred_digit] * 100

    fig = plt.figure(figsize=(11.5, 5.6), facecolor="#141413")
    ax = fig.add_axes([0, 0, 1, 1], facecolor="#141413")
    ax.set_xlim(0, 11.5)
    ax.set_ylim(0, 5.6)
    ax.axis("off")

    ax.text(
        0.35,
        5.22,
        "dataset: MNIST (70k)  |  28x28 thickened input",
        color="#a3a39e",
        fontsize=9,
        fontfamily="monospace",
    )
    ax.text(
        6.1,
        5.22,
        f"loss: {FINAL_LOSS:.3f}  |  test acc: {TEST_ACC:.1f}%",
        color="#e07a5f",
        fontsize=9,
        fontfamily="monospace",
    )
    ax.text(
        9.4,
        5.22,
        f"pred: {pred_digit} ({confidence:.0f}%)",
        color="#4ade80",
        fontsize=10.5,
        fontweight="bold",
        fontfamily="monospace",
    )

    ax_img = fig.add_axes([0.025, 0.30, 0.19, 0.42])
    ax_img.imshow(input_28x28, cmap="gray", vmin=0, vmax=1)
    ax_img.set_xticks([])
    ax_img.set_yticks([])
    for spine in ax_img.spines.values():
        spine.set_edgecolor("#4ade80")
        spine.set_linewidth(1.6)
    ax.text(
        1.38,
        1.32,
        "Thickened & Centered\n28x28 ANN Input",
        color="#d4d4d0",
        fontsize=8.5,
        ha="center",
        fontfamily="monospace",
    )

    in_idx = np.linspace(0, 783, 28, dtype=int)
    in_y = np.linspace(4.7, 0.75, len(in_idx))
    in_x = np.full_like(in_y, 3.1)

    h1_idx = np.linspace(0, 127, 32, dtype=int)
    h1_y = np.linspace(4.75, 0.7, len(h1_idx))
    h1_x = np.full_like(h1_y, 5.05)

    h2_idx = np.linspace(0, 63, 24, dtype=int)
    h2_y = np.linspace(4.65, 0.8, len(h2_idx))
    h2_x = np.full_like(h2_y, 6.9)

    out_y = np.linspace(4.55, 0.9, 10)
    out_x = np.full_like(out_y, 8.65)

    h1_norm = h1 / (np.max(h1) + 1e-6)
    h2_norm = h2 / (np.max(h2) + 1e-6)

    for i_pos, f_i in enumerate(in_idx):
        val = x_vec[f_i]
        if val < 0.15:
            continue
        for j_pos, h_i in enumerate(h1_idx):
            if h1_norm[h_i] < 0.25:
                continue
            w = W1[f_i, h_i]
            sig = val * h1_norm[h_i] * abs(w)
            if sig > 0.08:
                ax.plot(
                    [in_x[i_pos], h1_x[j_pos]],
                    [in_y[i_pos], h1_y[j_pos]],
                    color="#f59e0b" if w > 0 else "#38bdf8",
                    alpha=min(0.65, sig * 2.2),
                    linewidth=0.7,
                    zorder=1,
                )

    for j_pos, h1_i in enumerate(h1_idx):
        if h1_norm[h1_i] < 0.2:
            continue
        for k_pos, h2_i in enumerate(h2_idx):
            if h2_norm[h2_i] < 0.2:
                continue
            w = W2[h1_i, h2_i]
            sig = h1_norm[h1_i] * h2_norm[h2_i] * abs(w)
            if sig > 0.12:
                ax.plot(
                    [h1_x[j_pos], h2_x[k_pos]],
                    [h1_y[j_pos], h2_y[k_pos]],
                    color="#fbbf24" if w > 0 else "#60a5fa",
                    alpha=min(0.65, sig * 1.6),
                    linewidth=0.75,
                    zorder=1,
                )

    for k_pos, h2_i in enumerate(h2_idx):
        if h2_norm[h2_i] < 0.15:
            continue
        for o_i in range(10):
            w = W3[h2_i, o_i]
            sig = h2_norm[h2_i] * abs(w)
            if sig > 0.2:
                is_win = o_i == pred_digit
                ax.plot(
                    [h2_x[k_pos], out_x[o_i]],
                    [h2_y[k_pos], out_y[o_i]],
                    color="#4ade80" if is_win else "#d97706",
                    alpha=min(0.85, sig * 0.7) if is_win else 0.15,
                    linewidth=1.2 if is_win else 0.55,
                    zorder=1,
                )

    for i_pos, f_i in enumerate(in_idx):
        ax.scatter(
            in_x[i_pos],
            in_y[i_pos],
            s=35,
            color=plt.cm.gray(x_vec[f_i]),
            edgecolors="#787872",
            linewidth=0.7,
            zorder=3,
        )

    for j_pos, h1_i in enumerate(h1_idx):
        act = h1_norm[h1_i]
        ax.scatter(
            h1_x[j_pos],
            h1_y[j_pos],
            s=36,
            color=plt.cm.YlOrBr(0.15 + 0.85 * act) if act > 0.05 else "#262624",
            edgecolors="#f59e0b" if act > 0.3 else "#575752",
            linewidth=0.7,
            zorder=3,
        )

    for k_pos, h2_i in enumerate(h2_idx):
        act = h2_norm[h2_i]
        ax.scatter(
            h2_x[k_pos],
            h2_y[k_pos],
            s=42,
            color=plt.cm.YlOrBr(0.15 + 0.85 * act) if act > 0.05 else "#262624",
            edgecolors="#f59e0b" if act > 0.3 else "#575752",
            linewidth=0.8,
            zorder=3,
        )

    for o_i in range(10):
        prob = probs[o_i]
        is_win = o_i == pred_digit
        ax.scatter(
            out_x[o_i],
            out_y[o_i],
            s=130,
            color="#f59e0b" if is_win else "#262624",
            edgecolors="#fef08a" if is_win else "#787872",
            linewidth=1.3,
            zorder=4,
        )
        ax.text(
            out_x[o_i],
            out_y[o_i],
            str(o_i),
            color="#141413" if is_win else "#e5e5e0",
            fontsize=8.5,
            fontweight="bold",
            ha="center",
            va="center",
            zorder=5,
            fontfamily="monospace",
        )

        bar_x = 9.0
        max_w = 1.65
        ax.barh(out_y[o_i], max_w, left=bar_x, height=0.18, color="#262624", zorder=2)
        ax.barh(
            out_y[o_i],
            max_w * prob,
            left=bar_x,
            height=0.18,
            color="#4ade80" if is_win else "#78716c",
            zorder=3,
        )
        ax.text(
            bar_x + max_w + 0.1,
            out_y[o_i],
            f"{prob*100:4.1f}%",
            color="#4ade80" if is_win else "#a3a39e",
            fontsize=8,
            va="center",
            fontfamily="monospace",
        )

    ax.text(3.1, 0.22, "in · 784", color="#a3a39e", fontsize=8.5, ha="center", fontfamily="monospace")
    ax.text(5.05, 0.22, "hidden 1 · 128", color="#f59e0b", fontsize=8.5, ha="center", fontfamily="monospace")
    ax.text(6.9, 0.22, "hidden 2 · 64", color="#f59e0b", fontsize=8.5, ha="center", fontfamily="monospace")
    ax.text(8.65, 0.22, "out · 10", color="#4ade80", fontsize=8.5, ha="center", fontfamily="monospace")

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, facecolor="#141413")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

In [6]:
# ==========================================
# 4. GRADIO APP INTERFACE
# ==========================================
def analyze_handwriting(sketch_data):
    input_28x28 = preprocess_drawing(sketch_data)
    if input_28x28 is None:
        input_28x28 = SAMPLE_9
    x_vec, h1, h2, probs = forward_pass(input_28x28)
    viz_img = render_network_visual(input_28x28, x_vec, h1, h2, probs)
    pred = int(np.argmax(probs))
    return viz_img, f"Predicted Digit: {pred}   |   Confidence: {probs[pred]*100:.2f}%"


with gr.Blocks(theme=gr.themes.Base(), title="MNIST Live ANN Visualizer") as demo:
    gr.Markdown("""
        #Live MNIST Artificial Neural Network Visualizer (784 → 128 → 64 → 10)
        **Zeal College of Engineering & Research | Dept. of AI & DS | CIE-II Activity**
        Trained on **300,000 augmented 28×28 MNIST images** in pure NumPy. Draw any digit (**0–9**) on the left pad!
        """)

    with gr.Row():
        with gr.Column(scale=1):
            sketchpad = gr.Sketchpad(
                label="Write Any Digit Here (0–9)",
                type="numpy",
                height=360,
                width=360,
                brush=gr.Brush(default_size=26, colors=["#000000"]),
            )
            with gr.Row():
                clear_btn = gr.ClearButton([sketchpad], value="Clear")
                run_btn = gr.Button("Predict & Visualize", variant="primary")
            result_banner = gr.Textbox(
                label="ANN Classification Output",
                value="Draw a digit to see live forward propagation!",
                interactive=False,
            )

        with gr.Column(scale=3):
            network_plot = gr.Image(
                label="Real-Time Neural Firing (784 → 128 ReLU → 64 ReLU → 10 Softmax)",
                type="pil",
            )

    demo.load(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])
    run_btn.click(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])
    sketchpad.change(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])

if __name__ == "__main__":
    demo.launch()

C:\Users\Dell\AppData\Local\Temp\ipykernel_20812\4079143659.py:14: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), title="MNIST Live ANN Visualizer") as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
